# Aula 08 - Notebook: Sistemas Especialistas — Base de Conhecimento e Regras de Diagnóstico

Neste notebook implementamos a arquitetura de **Base de Conhecimento Industrial** orientada a objetos para a **Estação de Reabastecimento de Hidrogênio** (SCADA-Core). Estruturamos fatos de sensores, regras de produção em Cláusulas de Horn, mecanismos de verificação de consistência e exportação de relatórios tabulares de diagnóstico de causa-raiz.

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass, field
from typing import List, Set, Dict, Any, Optional
import time

@dataclass
class Fato:
    nome: str
    valor: bool
    descricao: str
    fonte: str = "SENSOR"
    timestamp: float = field(default_factory=time.time)

@dataclass
class RegraDiagnostico:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    severidade: str
    prioridade: int
    tempo_resposta_max_s: float
    acao_corretiva: str

class BaseConhecimentoSCADA:
    def __init__(self):
        self.regras: List[RegraDiagnostico] = []
        self._indice_antecedentes: Dict[str, List[RegraDiagnostico]] = {}

    def adicionar_regra(
        self, id_regra: str, antecedentes: List[str], consequente: str,
        descricao: str, severidade: str = "ALTA", prioridade: int = 5,
        tempo_max_s: float = 5.0, acao: str = "Verificar malha"
    ):
        regra = RegraDiagnostico(
            id_regra=id_regra,
            antecedentes=set(antecedentes),
            consequente=consequente,
            descricao_diagnostico=descricao,
            severidade=severidade,
            prioridade=prioridade,
            tempo_resposta_max_s=tempo_max_s,
            acao_corretiva=acao
        )
        self.regras.append(regra)
        
        for ant in antecedentes:
            if ant not in self._indice_antecedentes:
                self._indice_antecedentes[ant] = []
            self._indice_antecedentes[ant].append(regra)

    def obter_regras_por_fato(self, fato_nome: str) -> List[RegraDiagnostico]:
        return self._indice_antecedentes.get(fato_nome, [])

    def exportar_catalogo(self) -> List[Dict[str, Any]]:
        catalogo = []
        for r in sorted(self.regras, key=lambda x: x.prioridade, reverse=True):
            catalogo.append({
                "ID": r.id_regra,
                "Prioridade": r.prioridade,
                "Severidade": r.severidade,
                "SE (Antecedentes)": " AND ".join(sorted(r.antecedentes)),
                "ENTÃO (Consequente)": r.consequente,
                "Diagnóstico": r.descricao_diagnostico,
                "Ação Corretiva Recomendada": r.acao_corretiva
            })
        return catalogo

    def diagnosticar(self, sintomas_observados: Set[str]) -> List[Dict[str, Any]]:
        diagnosticos = []
        regras_ordenadas = sorted(self.regras, key=lambda x: x.prioridade, reverse=True)
        for regra in regras_ordenadas:
            if regra.antecedentes.issubset(sintomas_observados):
                diagnosticos.append({
                    "id": regra.id_regra,
                    "causa": regra.descricao_diagnostico,
                    "severidade": regra.severidade,
                    "acao": regra.acao_corretiva
                })
        return diagnosticos

bc = BaseConhecimentoSCADA()

bc.adicionar_regra(
    "R-01", ["FT101_CORRENTE_BAIXO"], "FALHA_ELETRICA_FT101",
    "Cabo Rompido / Falha Elétrica de Sensor", "ALTA", 8, 5.0,
    "ALTA: Manutenção corretiva na malha 4-20mA do FT-101"
)
bc.adicionar_regra(
    "R-02", ["FT101_FLUXO_LOW", "PT101_PRESSAO_LOW"], "CAVITACAO_BOMBA",
    "Cavitação ou Falha na Bomba de Alimentação", "CRÍTICA", 9, 2.0,
    "CRÍTICA: Desligar bomba e verificar sucção/válvula de montante"
)
bc.adicionar_regra(
    "R-03", ["PT101_HIGH", "LT101_HIGH"], "BLOQUEIO_SAIDA",
    "Bloqueio na Linha de Saída / Sobretensão", "CRÍTICA", 9, 1.0,
    "CRÍTICA: Abrir válvula de alívio e fechar alimentação XV-101"
)
bc.adicionar_regra(
    "R-04", ["AT101_PH_BAIXO"], "DESVIO_DOSAGEM",
    "Desvio de Dosagem Química (Acidificação)", "MÉDIA", 5, 10.0,
    "MÉDIA: Incrementar vazão da válvula de dosagem de base FCV-102"
)
bc.adicionar_regra(
    "R-05", ["AT101_GAS_HIGH", "XV301_OPEN"], "VAZAMENTO_H2_DISP",
    "Vazamento de H2 na Zona do Dispensador", "CRÍTICA", 10, 0.5,
    "CRÍTICA (SIL 3): Fechar válvula XV-301 e disparar Trip ESD-100"
)
bc.adicionar_regra(
    "R-06", ["PT101_HIGH", "TT101_HIGH"], "SOBREPRESSAO_BANCO_H2",
    "Pressão e Temperatura Excessivas no Armazenamento", "EMERGÊNCIA", 10, 0.2,
    "EMERGÊNCIA: Despressurizar banco para alívio/venting e purga"
)

print("=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (ESTAÇÃO DE H2) ===")
print(formatar_tabela(bc.exportar_catalogo()))

assert len(bc.regras) == 6
assert len(bc.obter_regras_por_fato("PT101_HIGH")) >= 2
print("\n[OK] Base de Conhecimento estruturada, indexada e validada com sucesso!")

=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (ESTAÇÃO DE H2) ===
ID   | Prioridade | Severidade | SE (Antecedentes)            | ENTÃO (Consequente)      | Diagnóstico                                     | Ação Corretiva Recomendada
-----+------------+------------+------------------------------+--------------------------+-------------------------------------------------+-------------------------------------------------------------
R-05 | 10         | CRÍTICA    | AT101_GAS_HIGH AND XV301_OPEN| VAZAMENTO_H2_DISP        | Vazamento de H2 na Zona do Dispensador          | CRÍTICA (SIL 3): Fechar válvula XV-301 e disparar Trip ESD-100
R-06 | 10         | EMERGÊNCIA | PT101_HIGH AND TT101_HIGH    | SOBREPRESSAO_BANCO_H2    | Pressão e Temperatura Excessivas no Armazenamento| EMERGÊNCIA: Despressurizar banco para alívio/venting e purga
R-02 | 9          | CRÍTICA    | FT101_FLUXO_LOW AND PT101_LOW| CAVITACAO_BOMBA          | Cavitação ou Falha na Bomba de Alimentação      | CRÍ

In [2]:
cenarios_diagnostico = {
    "Cenário A: Operação Nominal": set(),
    "Cenário B: Falha Elétrica FT-101": {"FT101_CORRENTE_BAIXO"},
    "Cenário C: Bloqueio Downstream": {"PT101_HIGH", "LT101_HIGH"},
    "Cenário D: Cavitação de Bomba": {"FT101_FLUXO_LOW", "PT101_PRESSAO_LOW"},
    "Cenário E: Vazamento de Hidrogênio": {"AT101_GAS_HIGH", "XV301_OPEN"}
}

print(f"{'Cenário Operacional':<34} | {'Causa Raiz Diagnosticada':<45} | {'Severidade':<12} | {'Ação Corretiva'}")
print("-" * 135)

for nome, sintomas in cenarios_diagnostico.items():
    resultados = bc.diagnosticar(sintomas)
    if not resultados:
        print(f"{nome:<34} | {'Nenhuma falha identificada (Nominal)':<45} | {'NORMAL':<12} | {'Manter monitoramento contínuo'}")
    else:
        for res in resultados:
            print(f"{nome:<34} | {res['causa']:<45} | {res['severidade']:<12} | {res['acao']}")

print("\n[OK] Simulação de inferência de causa-raiz executada com 100% de sucesso!")

=== RELATÓRIO DE DIAGNÓSTICO DO SISTEMA ESPECIALISTA ===
Cenário Operacional                | Causa Raiz Diagnosticada                      | Severidade   | Ação Corretiva
-----------------------------------+-----------------------------------------------+--------------+-------------------------------------------------------------
Cenário A: Operação Nominal        | Nenhuma falha identificada (Nominal)          | NORMAL       | Manter monitoramento contínuo
Cenário B: Falha Elétrica FT-101   | Cabo Rompido / Falha Elétrica de Sensor       | ALTA         | ALTA: Manutenção corretiva na malha 4-20mA do FT-101        
Cenário C: Bloqueio Downstream     | Bloqueio na Linha de Saída / Sobretensão      | CRÍTICA      | CRÍTICA: Abrir válvula de alívio e fechar alimentação XV-101
Cenário D: Cavitação de Bomba      | Cavitação ou Falha na Bomba de Alimentação    | CRÍTICA      | CRÍTICA: Desligar bomba e verificar sucção/válvula de montante
Cenário E: Vazamento de Hidrogênio | Vazamento de H2